# CogAttention — Attentional Blink

**Track:** Attention — Attention Capacity
**Benchmark:** CogAttention v1.0
**Task:** blink

---

## Methodology

Tests attentional blink — identifying two targets in a rapid serial word stream. When the second target appears shortly after the first, humans exhibit a temporary inability to process it. We test whether LLMs show analogous temporal bottlenecks. Based on RSVP (Raymond et al., 1992).

### Cognitive Science Grounding

- **Attentional Blink** (Raymond et al., 1992): after detecting a first target in a rapid stream, humans temporarily cannot process a second target appearing 200-500ms later
- We test whether LLMs show analogous temporal bottlenecks in sequential processing

### Difficulty Scaling

Easy: 20-item stream, lag 8 | Medium: 30 items, lag 5 | Hard: 40 items, lag 3 | Expert: 60 items, lag 2 | Frontier: 80 items, lag 1

### Scoring

Two assertions per item — one for T1 (ALL-CAPS target), one for T2 (hyphenated number-word). Both must be identified correctly.

All instances are procedurally generated from a seed with programmatic ground truth.
No static datasets. 7 layers of contamination resistance including canary strings,
zero lexical overlap (Semantic NIAH), and seed-based regeneration.

---

`<!-- COGATTENTION-BENCH-CANARY-C4FEA857B4BD -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Attention Capacity
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re

def extract_answer_block(response):
    for pat in [r"ANSWER:\s*(.*)", r"Answer:\s*(.*)", r"answer:\s*(.*)"]:
        match = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return response.strip()

def extract_numbered_answers(response):
    answer_block = extract_answer_block(response)
    results = {}
    matches = re.findall(
        r"(\d+)\s*[.):\-]\s*(.+?)(?=\n\d+\s*[.):\-]|\Z)",
        answer_block, re.DOTALL,
    )
    for num, val in matches:
        results[num] = val.strip().rstrip(".")
    return results

def extract_list_items(response):
    answer_block = extract_answer_block(response)
    bullets = re.findall(r"[-\u2022]\s*(.+?)(?:\n|$)", answer_block)
    if bullets:
        return [b.strip().rstrip(".") for b in bullets]
    numeric_items = re.findall(
        r'[\$]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*(?:\xb0[CF]|mg/L|%|\$))?',
        answer_block,
    )
    if numeric_items and len(numeric_items) >= 2:
        return [x.strip() for x in numeric_items]
    if "," in answer_block:
        items = [x.strip().rstrip(".") for x in answer_block.split(",")]
        return [x for x in items if x]
    lines = [l.strip().rstrip(".") for l in answer_block.split("\n") if l.strip()]
    return lines if lines else ([answer_block] if answer_block else [])

def extract_person_item_pairs(response):
    answer_block = extract_answer_block(response)
    results = {}
    for pat in [
        r"[-\u2022]?\s*(\w+)\s*:\s*(.+?)(?:\n|$)",
        r"[-\u2022]?\s*(\w+)\s+holds?\s+(?:a\s+)?(.+?)(?:\n|$)",
    ]:
        matches = re.findall(pat, answer_block, re.IGNORECASE)
        if matches:
            for name, item in matches:
                results[name.strip()] = item.strip().rstrip(".")
            break
    return results

def fuzzy_value_match(predicted, gold):
    pred_clean = re.sub(r"\s+", " ", predicted.strip().lower())
    gold_clean = re.sub(r"\s+", " ", gold.strip().lower())
    if pred_clean == gold_clean:
        return True
    if gold_clean in pred_clean:
        return True
    try:
        pred_num = float(re.sub(r"[,$%\xb0]", "", predicted))
        gold_num = float(re.sub(r"[,$%\xb0]", "", gold))
        return pred_num == gold_num
    except (ValueError, TypeError):
        pass
    return False

def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_blink(response, gold, kbench):
    for target_key in ["t1", "t2"]:
        gold_val = gold[target_key]
        label = target_key.upper()
        pattern = rf"(?i){re.escape(label)}\s*[:\-=]\s*.*{_escape_for_regex(gold_val)}"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"{label} should be '{gold_val}'"
        )


print("CogAttention helpers loaded")
print(f"Task types: ['blink']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_blink")
def cogattention_blink(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention blink task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_blink(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "blink_easy_000",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. chapter\n2. stairway\n3. monument\n4. chimney\n5. EMERALD\n6. balcony\n7. pavilion\n8. corridor\n9. curtain\n10. fountain\n11. pattern\n12. table\n13. forty-five\n14. street\n15. bridge\n16. station\n17. lantern\n18. market\n19. village\n20. window\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"forty-five\"}"
 },
 {
  "task_id": "blink_easy_001",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. highway\n2. blanket\n3. market\n4. signal\n5. DIAMOND\n6. column\n7. river\n8. pattern\n9. kitchen\n10. curtain\n11. corner\n12. bridge\n13. nine-million\n14. corridor\n15. garden\n16. doorway\n17. chimney\n18. factory\n19. library\n20. monument\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"DIAMOND\", \"t2\": \"nine-million\"}"
 },
 {
  "task_id": "blink_easy_002",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. window\n2. balcony\n3. gallery\n4. district\n5. RUBY\n6. corner\n7. pattern\n8. stairway\n9. surface\n10. fountain\n11. blanket\n12. river\n13. nine-million\n14. chapter\n15. curtain\n16. cabinet\n17. library\n18. harbor\n19. garden\n20. platform\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"RUBY\", \"t2\": \"nine-million\"}"
 },
 {
  "task_id": "blink_easy_003",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. balcony\n2. table\n3. morning\n4. fountain\n5. TOPAZ\n6. river\n7. kitchen\n8. cabinet\n9. market\n10. highway\n11. curtain\n12. chamber\n13. four-thousand\n14. library\n15. factory\n16. ceiling\n17. stairway\n18. platform\n19. doorway\n20. surface\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"TOPAZ\", \"t2\": \"four-thousand\"}"
 },
 {
  "task_id": "blink_easy_004",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. library\n2. surface\n3. curtain\n4. doorway\n5. TOPAZ\n6. bridge\n7. passage\n8. terrace\n9. blanket\n10. building\n11. gallery\n12. garden\n13. thirty-six\n14. window\n15. factory\n16. balcony\n17. chimney\n18. column\n19. ceiling\n20. kitchen\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"TOPAZ\", \"t2\": \"thirty-six\"}"
 },
 {
  "task_id": "blink_easy_005",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. kitchen\n2. evening\n3. monument\n4. bridge\n5. TOPAZ\n6. shelter\n7. factory\n8. corner\n9. stairway\n10. curtain\n11. table\n12. highway\n13. twenty-eight\n14. pattern\n15. passage\n16. ceiling\n17. gallery\n18. surface\n19. cabinet\n20. blanket\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"TOPAZ\", \"t2\": \"twenty-eight\"}"
 },
 {
  "task_id": "blink_easy_006",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. passage\n2. gallery\n3. column\n4. highway\n5. RUBY\n6. curtain\n7. shelter\n8. street\n9. district\n10. cabinet\n11. surface\n12. table\n13. twenty-eight\n14. chimney\n15. library\n16. doorway\n17. bridge\n18. village\n19. evening\n20. balcony\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"RUBY\", \"t2\": \"twenty-eight\"}"
 },
 {
  "task_id": "blink_easy_007",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. curtain\n2. column\n3. terrace\n4. gallery\n5. AMETHYST\n6. bridge\n7. ceiling\n8. table\n9. balcony\n10. river\n11. library\n12. chapter\n13. nine-million\n14. kitchen\n15. evening\n16. window\n17. market\n18. harbor\n19. street\n20. platform\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"AMETHYST\", \"t2\": \"nine-million\"}"
 },
 {
  "task_id": "blink_medium_008",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. shelter\n2. chimney\n3. curtain\n4. morning\n5. stairway\n6. market\n7. DIAMOND\n8. pavilion\n9. library\n10. kitchen\n11. balcony\n12. seven-hundred\n13. street\n14. monument\n15. chamber\n16. garden\n17. platform\n18. ceiling\n19. terrace\n20. building\n21. lantern\n22. cabinet\n23. surface\n24. fountain\n25. river\n26. district\n27. passage\n28. factory\n29. pattern\n30. bridge\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"DIAMOND\", \"t2\": \"seven-hundred\"}"
 },
 {
  "task_id": "blink_medium_009",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. district\n2. factory\n3. surface\n4. ceiling\n5. window\n6. platform\n7. EMERALD\n8. building\n9. village\n10. chapter\n11. table\n12. sixty-three\n13. monument\n14. corridor\n15. shelter\n16. gallery\n17. garden\n18. corner\n19. doorway\n20. stairway\n21. evening\n22. lantern\n23. curtain\n24. column\n25. chimney\n26. highway\n27. pavilion\n28. bridge\n29. terrace\n30. library\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"sixty-three\"}"
 },
 {
  "task_id": "blink_medium_010",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. ceiling\n2. chapter\n3. stairway\n4. passage\n5. platform\n6. evening\n7. morning\n8. building\n9. terrace\n10. EMERALD\n11. blanket\n12. fountain\n13. market\n14. river\n15. sixty-three\n16. corner\n17. garden\n18. doorway\n19. chimney\n20. library\n21. table\n22. kitchen\n23. gallery\n24. factory\n25. pattern\n26. column\n27. corridor\n28. monument\n29. signal\n30. station\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"sixty-three\"}"
 },
 {
  "task_id": "blink_medium_011",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. station\n2. corner\n3. market\n4. gallery\n5. balcony\n6. corridor\n7. monument\n8. highway\n9. pattern\n10. DIAMOND\n11. morning\n12. platform\n13. garden\n14. lantern\n15. sixty-three\n16. stairway\n17. library\n18. chimney\n19. table\n20. surface\n21. district\n22. signal\n23. pavilion\n24. column\n25. blanket\n26. chamber\n27. street\n28. bridge\n29. passage\n30. ceiling\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"DIAMOND\", \"t2\": \"sixty-three\"}"
 },
 {
  "task_id": "blink_medium_012",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. building\n2. district\n3. highway\n4. chamber\n5. river\n6. kitchen\n7. cabinet\n8. pattern\n9. evening\n10. EMERALD\n11. market\n12. terrace\n13. harbor\n14. chapter\n15. seven-hundred\n16. doorway\n17. table\n18. morning\n19. library\n20. balcony\n21. stairway\n22. street\n23. bridge\n24. corner\n25. fountain\n26. blanket\n27. gallery\n28. factory\n29. ceiling\n30. signal\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"seven-hundred\"}"
 },
 {
  "task_id": "blink_medium_013",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. balcony\n2. lantern\n3. garden\n4. surface\n5. shelter\n6. factory\n7. RUBY\n8. platform\n9. bridge\n10. cabinet\n11. market\n12. sixty-three\n13. river\n14. blanket\n15. evening\n16. street\n17. library\n18. doorway\n19. district\n20. monument\n21. signal\n22. morning\n23. passage\n24. window\n25. highway\n26. column\n27. corridor\n28. building\n29. stairway\n30. gallery\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"RUBY\", \"t2\": \"sixty-three\"}"
 },
 {
  "task_id": "blink_medium_014",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. chapter\n2. building\n3. kitchen\n4. platform\n5. library\n6. evening\n7. GARNET\n8. balcony\n9. lantern\n10. fountain\n11. blanket\n12. forty-five\n13. table\n14. corner\n15. factory\n16. curtain\n17. village\n18. river\n19. corridor\n20. street\n21. surface\n22. terrace\n23. harbor\n24. doorway\n25. garden\n26. chamber\n27. gallery\n28. market\n29. station\n30. highway\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"GARNET\", \"t2\": \"forty-five\"}"
 },
 {
  "task_id": "blink_medium_015",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. corner\n2. library\n3. harbor\n4. village\n5. curtain\n6. pattern\n7. GARNET\n8. cabinet\n9. kitchen\n10. column\n11. terrace\n12. forty-five\n13. gallery\n14. ceiling\n15. market\n16. passage\n17. fountain\n18. lantern\n19. chapter\n20. shelter\n21. signal\n22. station\n23. street\n24. doorway\n25. blanket\n26. morning\n27. chimney\n28. bridge\n29. building\n30. stairway\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"GARNET\", \"t2\": \"forty-five\"}"
 },
 {
  "task_id": "blink_hard_016",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): an ALL-CAPS animal name (e.g., ELEPHANT), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. chimney\n2. chapter\n3. stairway\n4. lantern\n5. signal\n6. harbor\n7. station\n8. column\n9. table\n10. morning\n11. factory\n12. terrace\n13. library\n14. curtain\n15. surface\n16. doorway\n17. ceiling\n18. DIAMOND\n19. river\n20. pattern\n21. PENGUIN\n22. kitchen\n23. platform\n24. blanket\n25. window\n26. building\n27. corner\n28. pavilion\n29. bridge\n30. passage\n31. gallery\n32. chamber\n33. district\n34. evening\n35. monument\n36. market\n37. cabinet\n38. street\n39. village\n40. highway\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"DIAMOND\", \"t2\": \"PENGUIN\"}"
 },
 {
  "task_id": "blink_hard_017",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): an ALL-CAPS animal name (e.g., ELEPHANT), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. market\n2. pattern\n3. signal\n4. morning\n5. factory\n6. highway\n7. window\n8. platform\n9. balcony\n10. stairway\n11. shelter\n12. monument\n13. chamber\n14. pavilion\n15. village\n16. EMERALD\n17. ceiling\n18. column\n19. PENGUIN\n20. evening\n21. district\n22. fountain\n23. curtain\n24. corner\n25. bridge\n26. library\n27. terrace\n28. passage\n29. building\n30. street\n31. harbor\n32. station\n33. kitchen\n34. river\n35. corridor\n36. chimney\n37. table\n38. gallery\n39. blanket\n40. lantern\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"PENGUIN\"}"
 },
 {
  "task_id": "blink_hard_018",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): an ALL-CAPS animal name (e.g., ELEPHANT), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. harbor\n2. platform\n3. signal\n4. window\n5. river\n6. corridor\n7. chapter\n8. building\n9. station\n10. evening\n11. OPAL\n12. stairway\n13. library\n14. OCTOPUS\n15. curtain\n16. fountain\n17. surface\n18. market\n19. bridge\n20. chamber\n21. doorway\n22. factory\n23. pavilion\n24. kitchen\n25. shelter\n26. terrace\n27. blanket\n28. garden\n29. street\n30. cabinet\n31. monument\n32. district\n33. morning\n34. lantern\n35. gallery\n36. chimney\n37. highway\n38. village\n39. ceiling\n40. passage\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"OPAL\", \"t2\": \"OCTOPUS\"}"
 },
 {
  "task_id": "blink_hard_019",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): an ALL-CAPS animal name (e.g., ELEPHANT), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. cabinet\n2. river\n3. fountain\n4. doorway\n5. chamber\n6. morning\n7. corner\n8. passage\n9. market\n10. corridor\n11. harbor\n12. RUBY\n13. lantern\n14. factory\n15. LEOPARD\n16. gallery\n17. platform\n18. window\n19. curtain\n20. blanket\n21. garden\n22. chimney\n23. chapter\n24. library\n25. terrace\n26. signal\n27. balcony\n28. village\n29. district\n30. bridge\n31. pattern\n32. station\n33. evening\n34. street\n35. highway\n36. surface\n37. table\n38. monument\n39. column\n40. ceiling\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"RUBY\", \"t2\": \"LEOPARD\"}"
 },
 {
  "task_id": "blink_hard_020",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): an ALL-CAPS animal name (e.g., ELEPHANT), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. factory\n2. chapter\n3. market\n4. signal\n5. pavilion\n6. station\n7. lantern\n8. ceiling\n9. corridor\n10. TOPAZ\n11. chamber\n12. shelter\n13. ELEPHANT\n14. street\n15. platform\n16. garden\n17. kitchen\n18. blanket\n19. highway\n20. passage\n21. bridge\n22. window\n23. curtain\n24. table\n25. gallery\n26. surface\n27. pattern\n28. river\n29. fountain\n30. library\n31. chimney\n32. stairway\n33. cabinet\n34. building\n35. monument\n36. morning\n37. doorway\n38. balcony\n39. evening\n40. district\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"TOPAZ\", \"t2\": \"ELEPHANT\"}"
 },
 {
  "task_id": "blink_hard_021",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): an ALL-CAPS animal name (e.g., ELEPHANT), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. kitchen\n2. factory\n3. market\n4. highway\n5. gallery\n6. blanket\n7. cabinet\n8. chapter\n9. station\n10. AMETHYST\n11. fountain\n12. river\n13. LEOPARD\n14. table\n15. doorway\n16. pavilion\n17. terrace\n18. platform\n19. column\n20. corner\n21. bridge\n22. surface\n23. village\n24. chimney\n25. monument\n26. stairway\n27. lantern\n28. curtain\n29. library\n30. pattern\n31. district\n32. morning\n33. passage\n34. evening\n35. balcony\n36. building\n37. signal\n38. garden\n39. ceiling\n40. shelter\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"AMETHYST\", \"t2\": \"LEOPARD\"}"
 },
 {
  "task_id": "blink_hard_022",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): an ALL-CAPS animal name (e.g., ELEPHANT), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. garden\n2. kitchen\n3. evening\n4. gallery\n5. window\n6. passage\n7. chimney\n8. corner\n9. table\n10. chapter\n11. bridge\n12. factory\n13. platform\n14. monument\n15. stairway\n16. OPAL\n17. market\n18. lantern\n19. PEACOCK\n20. building\n21. column\n22. street\n23. chamber\n24. highway\n25. doorway\n26. library\n27. balcony\n28. pavilion\n29. cabinet\n30. curtain\n31. blanket\n32. morning\n33. village\n34. surface\n35. shelter\n36. terrace\n37. ceiling\n38. corridor\n39. pattern\n40. station\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"OPAL\", \"t2\": \"PEACOCK\"}"
 },
 {
  "task_id": "blink_hard_023",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): an ALL-CAPS animal name (e.g., ELEPHANT), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. pattern\n2. platform\n3. terrace\n4. pavilion\n5. stairway\n6. fountain\n7. column\n8. doorway\n9. table\n10. highway\n11. market\n12. passage\n13. monument\n14. EMERALD\n15. window\n16. chamber\n17. BUFFALO\n18. harbor\n19. chapter\n20. blanket\n21. signal\n22. chimney\n23. library\n24. bridge\n25. building\n26. cabinet\n27. station\n28. corner\n29. corridor\n30. street\n31. district\n32. curtain\n33. river\n34. morning\n35. shelter\n36. garden\n37. ceiling\n38. kitchen\n39. surface\n40. village\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"BUFFALO\"}"
 },
 {
  "task_id": "blink_expert_024",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. fountain\n2. ceiling\n3. gallery\n4. kitchen\n5. doorway\n6. signal\n7. library\n8. highway\n9. platform\n10. chimney\n11. stairway\n12. river\n13. street\n14. corridor\n15. GARNET\n16. surface\n17. twenty-eight\n18. lantern\n19. village\n20. building\n21. corner\n22. market\n23. blanket\n24. factory\n25. garden\n26. chapter\n27. district\n28. cabinet\n29. monument\n30. passage\n31. terrace\n32. window\n33. shelter\n34. morning\n35. evening\n36. harbor\n37. pattern\n38. station\n39. curtain\n40. table\n41. bridge\n42. chamber\n43. pavilion\n44. column\n45. balcony\n46. street\n47. column\n48. street\n49. ceiling\n50. station\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"GARNET\", \"t2\": \"twenty-eight\"}"
 },
 {
  "task_id": "blink_expert_025",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. signal\n2. chimney\n3. bridge\n4. harbor\n5. cabinet\n6. fountain\n7. ceiling\n8. monument\n9. morning\n10. kitchen\n11. village\n12. column\n13. OPAL\n14. pavilion\n15. nine-million\n16. lantern\n17. window\n18. doorway\n19. balcony\n20. corridor\n21. river\n22. platform\n23. building\n24. factory\n25. stairway\n26. gallery\n27. table\n28. terrace\n29. district\n30. corner\n31. shelter\n32. garden\n33. street\n34. passage\n35. blanket\n36. highway\n37. chapter\n38. evening\n39. chamber\n40. pattern\n41. market\n42. curtain\n43. library\n44. station\n45. surface\n46. signal\n47. evening\n48. curtain\n49. signal\n50. chimney\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"OPAL\", \"t2\": \"nine-million\"}"
 },
 {
  "task_id": "blink_expert_026",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. balcony\n2. column\n3. cabinet\n4. blanket\n5. shelter\n6. evening\n7. fountain\n8. doorway\n9. table\n10. market\n11. library\n12. lantern\n13. building\n14. highway\n15. street\n16. district\n17. station\n18. bridge\n19. corridor\n20. chamber\n21. terrace\n22. EMERALD\n23. stairway\n24. thirty-six\n25. monument\n26. river\n27. village\n28. pavilion\n29. chapter\n30. garden\n31. surface\n32. factory\n33. corner\n34. chimney\n35. harbor\n36. ceiling\n37. signal\n38. morning\n39. pattern\n40. platform\n41. kitchen\n42. gallery\n43. curtain\n44. window\n45. passage\n46. stairway\n47. street\n48. blanket\n49. market\n50. passage\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"thirty-six\"}"
 },
 {
  "task_id": "blink_expert_027",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. table\n2. surface\n3. chamber\n4. fountain\n5. evening\n6. station\n7. blanket\n8. passage\n9. kitchen\n10. river\n11. RUBY\n12. column\n13. twenty-eight\n14. library\n15. platform\n16. doorway\n17. factory\n18. terrace\n19. market\n20. lantern\n21. shelter\n22. highway\n23. monument\n24. signal\n25. corridor\n26. stairway\n27. ceiling\n28. bridge\n29. window\n30. garden\n31. gallery\n32. chapter\n33. curtain\n34. pattern\n35. village\n36. corner\n37. morning\n38. chimney\n39. district\n40. pavilion\n41. street\n42. balcony\n43. cabinet\n44. harbor\n45. building\n46. chamber\n47. highway\n48. station\n49. library\n50. window\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"RUBY\", \"t2\": \"twenty-eight\"}"
 },
 {
  "task_id": "blink_expert_028",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. village\n2. harbor\n3. table\n4. doorway\n5. factory\n6. station\n7. corridor\n8. garden\n9. balcony\n10. passage\n11. terrace\n12. pavilion\n13. monument\n14. ceiling\n15. OPAL\n16. shelter\n17. sixty-three\n18. building\n19. platform\n20. cabinet\n21. street\n22. lantern\n23. fountain\n24. highway\n25. district\n26. stairway\n27. pattern\n28. bridge\n29. gallery\n30. river\n31. library\n32. chapter\n33. curtain\n34. signal\n35. chamber\n36. evening\n37. surface\n38. corner\n39. market\n40. kitchen\n41. morning\n42. column\n43. blanket\n44. chimney\n45. window\n46. ceiling\n47. column\n48. pavilion\n49. building\n50. platform\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"OPAL\", \"t2\": \"sixty-three\"}"
 },
 {
  "task_id": "blink_expert_029",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. curtain\n2. evening\n3. pavilion\n4. corridor\n5. factory\n6. gallery\n7. morning\n8. surface\n9. fountain\n10. chapter\n11. market\n12. bridge\n13. GARNET\n14. river\n15. sixty-three\n16. district\n17. table\n18. pattern\n19. balcony\n20. harbor\n21. doorway\n22. stairway\n23. ceiling\n24. cabinet\n25. monument\n26. village\n27. building\n28. corner\n29. passage\n30. chimney\n31. station\n32. column\n33. chamber\n34. terrace\n35. garden\n36. window\n37. highway\n38. platform\n39. street\n40. blanket\n41. library\n42. lantern\n43. signal\n44. shelter\n45. kitchen\n46. river\n47. monument\n48. harbor\n49. evening\n50. pavilion\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"GARNET\", \"t2\": \"sixty-three\"}"
 },
 {
  "task_id": "blink_expert_030",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. chimney\n2. highway\n3. terrace\n4. market\n5. harbor\n6. table\n7. district\n8. signal\n9. corridor\n10. passage\n11. garden\n12. pavilion\n13. curtain\n14. doorway\n15. kitchen\n16. library\n17. street\n18. morning\n19. river\n20. station\n21. monument\n22. shelter\n23. RUBY\n24. surface\n25. seven-hundred\n26. corner\n27. chapter\n28. balcony\n29. stairway\n30. evening\n31. lantern\n32. bridge\n33. chamber\n34. ceiling\n35. gallery\n36. platform\n37. blanket\n38. village\n39. window\n40. fountain\n41. building\n42. factory\n43. cabinet\n44. pattern\n45. column\n46. passage\n47. surface\n48. library\n49. district\n50. curtain\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"RUBY\", \"t2\": \"seven-hundred\"}"
 },
 {
  "task_id": "blink_expert_031",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word (e.g., DIAMOND)\n  - Target 2 (T2): a hyphenated number-word (e.g., forty-five), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. chapter\n2. factory\n3. evening\n4. column\n5. street\n6. cabinet\n7. garden\n8. monument\n9. station\n10. chimney\n11. chamber\n12. ceiling\n13. signal\n14. AMETHYST\n15. corner\n16. thirty-six\n17. corridor\n18. kitchen\n19. morning\n20. terrace\n21. bridge\n22. blanket\n23. village\n24. building\n25. platform\n26. market\n27. curtain\n28. gallery\n29. passage\n30. highway\n31. library\n32. doorway\n33. pattern\n34. fountain\n35. pavilion\n36. surface\n37. window\n38. shelter\n39. river\n40. harbor\n41. stairway\n42. district\n43. lantern\n44. balcony\n45. table\n46. building\n47. highway\n48. surface\n49. river\n50. village\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"AMETHYST\", \"t2\": \"thirty-six\"}"
 },
 {
  "task_id": "blink_frontier_032",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): a hyphenated number-word (e.g., forty-five)\n  - Target 2 (T2): an ALL-CAPS word (e.g., DIAMOND), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. shelter\n2. pavilion\n3. table\n4. factory\n5. district\n6. lantern\n7. chapter\n8. harbor\n9. market\n10. doorway\n11. village\n12. bridge\n13. highway\n14. chamber\n15. pattern\n16. street\n17. sixty-three\n18. DIAMOND\n19. passage\n20. corner\n21. building\n22. surface\n23. curtain\n24. stairway\n25. evening\n26. fountain\n27. signal\n28. gallery\n29. garden\n30. chimney\n31. river\n32. morning\n33. column\n34. monument\n35. balcony\n36. blanket\n37. kitchen\n38. station\n39. window\n40. library\n41. terrace\n42. corridor\n43. platform\n44. ceiling\n45. cabinet\n46. surface\n47. district\n48. pavilion\n49. highway\n50. cabinet\n51. platform\n52. chamber\n53. market\n54. corner\n55. pavilion\n56. morning\n57. ceiling\n58. fountain\n59. window\n60. platform\n61. garden\n62. monument\n63. station\n64. lantern\n65. balcony\n66. garden\n67. street\n68. stairway\n69. corridor\n70. street\n71. surface\n72. shelter\n73. village\n74. chamber\n75. monument\n76. kitchen\n77. street\n78. cabinet\n79. chamber\n80. passage\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"sixty-three\", \"t2\": \"DIAMOND\"}"
 },
 {
  "task_id": "blink_frontier_033",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): a hyphenated number-word (e.g., forty-five)\n  - Target 2 (T2): an ALL-CAPS word (e.g., DIAMOND), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. chapter\n2. blanket\n3. surface\n4. station\n5. stairway\n6. ceiling\n7. library\n8. lantern\n9. pavilion\n10. curtain\n11. doorway\n12. table\n13. evening\n14. chimney\n15. balcony\n16. factory\n17. harbor\n18. corridor\n19. passage\n20. signal\n21. gallery\n22. highway\n23. district\n24. kitchen\n25. garden\n26. bridge\n27. village\n28. window\n29. street\n30. chamber\n31. column\n32. corner\n33. sixty-three\n34. OPAL\n35. cabinet\n36. morning\n37. market\n38. pattern\n39. river\n40. platform\n41. building\n42. fountain\n43. terrace\n44. monument\n45. shelter\n46. harbor\n47. pattern\n48. bridge\n49. chimney\n50. ceiling\n51. signal\n52. gallery\n53. street\n54. window\n55. market\n56. balcony\n57. pavilion\n58. ceiling\n59. morning\n60. surface\n61. pattern\n62. chamber\n63. balcony\n64. morning\n65. table\n66. table\n67. passage\n68. evening\n69. building\n70. window\n71. pattern\n72. chapter\n73. fountain\n74. shelter\n75. factory\n76. column\n77. evening\n78. platform\n79. kitchen\n80. window\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"sixty-three\", \"t2\": \"OPAL\"}"
 },
 {
  "task_id": "blink_frontier_034",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): a hyphenated number-word (e.g., forty-five)\n  - Target 2 (T2): an ALL-CAPS word (e.g., DIAMOND), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. window\n2. building\n3. fountain\n4. curtain\n5. chapter\n6. passage\n7. district\n8. shelter\n9. pavilion\n10. blanket\n11. morning\n12. station\n13. chamber\n14. column\n15. terrace\n16. garden\n17. doorway\n18. market\n19. highway\n20. cabinet\n21. twenty-eight\n22. OPAL\n23. surface\n24. street\n25. river\n26. library\n27. gallery\n28. corner\n29. ceiling\n30. lantern\n31. kitchen\n32. monument\n33. stairway\n34. factory\n35. evening\n36. bridge\n37. platform\n38. chimney\n39. table\n40. signal\n41. pattern\n42. village\n43. corridor\n44. balcony\n45. harbor\n46. street\n47. gallery\n48. street\n49. balcony\n50. window\n51. lantern\n52. lantern\n53. window\n54. evening\n55. surface\n56. gallery\n57. cabinet\n58. table\n59. factory\n60. evening\n61. village\n62. table\n63. fountain\n64. column\n65. lantern\n66. balcony\n67. window\n68. ceiling\n69. cabinet\n70. stairway\n71. river\n72. pattern\n73. lantern\n74. pattern\n75. fountain\n76. highway\n77. curtain\n78. factory\n79. shelter\n80. kitchen\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"twenty-eight\", \"t2\": \"OPAL\"}"
 },
 {
  "task_id": "blink_frontier_035",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): a hyphenated number-word (e.g., forty-five)\n  - Target 2 (T2): an ALL-CAPS word (e.g., DIAMOND), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. terrace\n2. kitchen\n3. village\n4. curtain\n5. station\n6. shelter\n7. lantern\n8. platform\n9. morning\n10. balcony\n11. ceiling\n12. garden\n13. river\n14. highway\n15. street\n16. blanket\n17. monument\n18. chapter\n19. doorway\n20. eighty-one\n21. OPAL\n22. factory\n23. pavilion\n24. harbor\n25. pattern\n26. evening\n27. table\n28. passage\n29. district\n30. chimney\n31. window\n32. chamber\n33. building\n34. stairway\n35. surface\n36. corridor\n37. fountain\n38. market\n39. corner\n40. signal\n41. library\n42. bridge\n43. cabinet\n44. column\n45. gallery\n46. kitchen\n47. stairway\n48. curtain\n49. library\n50. monument\n51. pattern\n52. platform\n53. curtain\n54. window\n55. morning\n56. stairway\n57. gallery\n58. station\n59. chamber\n60. column\n61. factory\n62. table\n63. surface\n64. platform\n65. surface\n66. chimney\n67. evening\n68. cabinet\n69. window\n70. kitchen\n71. ceiling\n72. window\n73. gallery\n74. highway\n75. corridor\n76. market\n77. surface\n78. curtain\n79. passage\n80. pavilion\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"eighty-one\", \"t2\": \"OPAL\"}"
 },
 {
  "task_id": "blink_frontier_036",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): a hyphenated number-word (e.g., forty-five)\n  - Target 2 (T2): an ALL-CAPS word (e.g., DIAMOND), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. gallery\n2. building\n3. corner\n4. column\n5. market\n6. garden\n7. chapter\n8. chimney\n9. morning\n10. street\n11. terrace\n12. passage\n13. table\n14. surface\n15. curtain\n16. platform\n17. evening\n18. shelter\n19. stairway\n20. balcony\n21. fountain\n22. doorway\n23. bridge\n24. corridor\n25. harbor\n26. chamber\n27. four-thousand\n28. DIAMOND\n29. river\n30. factory\n31. station\n32. library\n33. pavilion\n34. pattern\n35. monument\n36. village\n37. window\n38. ceiling\n39. blanket\n40. signal\n41. cabinet\n42. kitchen\n43. lantern\n44. district\n45. highway\n46. harbor\n47. pattern\n48. station\n49. market\n50. kitchen\n51. monument\n52. surface\n53. fountain\n54. pavilion\n55. harbor\n56. doorway\n57. surface\n58. pattern\n59. shelter\n60. blanket\n61. district\n62. library\n63. factory\n64. terrace\n65. village\n66. morning\n67. lantern\n68. evening\n69. library\n70. gallery\n71. chimney\n72. pavilion\n73. monument\n74. kitchen\n75. district\n76. lantern\n77. garden\n78. river\n79. highway\n80. river\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"four-thousand\", \"t2\": \"DIAMOND\"}"
 },
 {
  "task_id": "blink_frontier_037",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): a hyphenated number-word (e.g., forty-five)\n  - Target 2 (T2): an ALL-CAPS word (e.g., DIAMOND), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. corridor\n2. balcony\n3. gallery\n4. factory\n5. ceiling\n6. blanket\n7. district\n8. lantern\n9. stairway\n10. window\n11. morning\n12. pavilion\n13. shelter\n14. platform\n15. column\n16. evening\n17. passage\n18. doorway\n19. river\n20. garden\n21. kitchen\n22. library\n23. harbor\n24. building\n25. table\n26. curtain\n27. signal\n28. chapter\n29. street\n30. fountain\n31. cabinet\n32. surface\n33. thirty-six\n34. GARNET\n35. village\n36. market\n37. station\n38. terrace\n39. monument\n40. bridge\n41. corner\n42. pattern\n43. chamber\n44. highway\n45. chimney\n46. ceiling\n47. kitchen\n48. district\n49. corridor\n50. station\n51. platform\n52. corner\n53. building\n54. curtain\n55. passage\n56. doorway\n57. ceiling\n58. street\n59. garden\n60. market\n61. bridge\n62. factory\n63. market\n64. chamber\n65. passage\n66. balcony\n67. highway\n68. balcony\n69. surface\n70. factory\n71. lantern\n72. chapter\n73. bridge\n74. cabinet\n75. monument\n76. corner\n77. stairway\n78. table\n79. ceiling\n80. signal\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"thirty-six\", \"t2\": \"GARNET\"}"
 },
 {
  "task_id": "blink_frontier_038",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): a hyphenated number-word (e.g., forty-five)\n  - Target 2 (T2): an ALL-CAPS word (e.g., DIAMOND), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. gallery\n2. chamber\n3. window\n4. pavilion\n5. factory\n6. lantern\n7. pattern\n8. blanket\n9. kitchen\n10. market\n11. shelter\n12. terrace\n13. doorway\n14. corridor\n15. evening\n16. corner\n17. stairway\n18. monument\n19. garden\n20. seven-hundred\n21. EMERALD\n22. passage\n23. ceiling\n24. chimney\n25. cabinet\n26. signal\n27. river\n28. column\n29. harbor\n30. surface\n31. curtain\n32. district\n33. library\n34. platform\n35. village\n36. station\n37. building\n38. fountain\n39. table\n40. balcony\n41. highway\n42. street\n43. morning\n44. bridge\n45. chapter\n46. stairway\n47. village\n48. shelter\n49. lantern\n50. factory\n51. village\n52. chimney\n53. market\n54. doorway\n55. factory\n56. river\n57. ceiling\n58. cabinet\n59. passage\n60. harbor\n61. doorway\n62. morning\n63. evening\n64. lantern\n65. street\n66. highway\n67. passage\n68. shelter\n69. ceiling\n70. signal\n71. station\n72. monument\n73. harbor\n74. garden\n75. balcony\n76. surface\n77. village\n78. doorway\n79. lantern\n80. corner\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"seven-hundred\", \"t2\": \"EMERALD\"}"
 },
 {
  "task_id": "blink_frontier_039",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): a hyphenated number-word (e.g., forty-five)\n  - Target 2 (T2): an ALL-CAPS word (e.g., DIAMOND), appearing after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. garden\n2. passage\n3. building\n4. gallery\n5. village\n6. balcony\n7. pavilion\n8. factory\n9. fountain\n10. district\n11. monument\n12. platform\n13. cabinet\n14. harbor\n15. station\n16. table\n17. surface\n18. kitchen\n19. evening\n20. chamber\n21. column\n22. stairway\n23. curtain\n24. chapter\n25. lantern\n26. blanket\n27. corridor\n28. pattern\n29. morning\n30. window\n31. doorway\n32. four-thousand\n33. OPAL\n34. terrace\n35. ceiling\n36. bridge\n37. chimney\n38. street\n39. corner\n40. library\n41. market\n42. shelter\n43. signal\n44. highway\n45. river\n46. gallery\n47. monument\n48. district\n49. shelter\n50. monument\n51. library\n52. monument\n53. doorway\n54. curtain\n55. chapter\n56. street\n57. cabinet\n58. window\n59. surface\n60. signal\n61. street\n62. building\n63. chimney\n64. column\n65. garden\n66. platform\n67. factory\n68. table\n69. gallery\n70. district\n71. district\n72. morning\n73. river\n74. surface\n75. street\n76. lantern\n77. cabinet\n78. monument\n79. column\n80. harbor\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"four-thousand\", \"t2\": \"OPAL\"}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['blink']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "blink": cogattention_blink,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Attention Capacity")
